# Lab 07-04 — Golden-Set Regression Testing

**Track 07 · Evaluation** — freeze a fixed question set, and every pipeline change must not regress it.

Unit tests lock behavior; a **golden set** locks RAG quality. This lab builds a small golden set (8 questions over two invoice PDFs, from `src/evaluation/golden.py`), runs **two variants of the same pipeline that differ in exactly one knob** (`top_k = 3` vs `top_k = 1`), and enforces a regression contract: the reference pipeline (A) must not be beaten by the variant (B) on any reference-anchored metric, with a tolerance band for the noisier LLM-judge metric.

```text
golden questions (8, from evaluation/golden.py)
  -> one index over the invoice PDFs
  -> variant A: top_k = 3   (reference)
  -> variant B: top_k = 1   (the would-be regression)
  -> score faithfulness / contained / cosine per variant
  -> regression gate: A >= B (contained, cosine, faithful-tol)
```


## Setup

This notebook mirrors `src/curriculum/07-evaluation/04-golden-regression.py` exactly — the same verified code, split into cells. You can run it from anywhere: the imports cell walks up to the repo root and cd's into it, so every `Data/...` path resolves just like the lab script.

From the terminal, the lab runs as:

```bash
python src/curriculum/07-evaluation/04-golden-regression.py          # run + demo
python src/curriculum/07-evaluation/04-golden-regression.py --verify # verification gate
```

**LLM keys**: the lab generates with GroqLLM and judges with the local Ollama judge — `GROQ_API_KEY` must be in the repo-root `.env` (the imports cell loads it).

The next cell installs the lab-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# is a no-op safety net for fresh environments):
#   sentence-transformers -> local BGE embeddings (embeddings/bge.py)
#   faiss-cpu             -> the FAISS index (vectordb/faiss.py)
#   langchain-groq        -> GroqLLM (the generator, llms/groq.py)
#   python-dotenv         -> loads GROQ_API_KEY from the repo-root .env
#   pypdf                 -> the invoice PDF loader (loaders/pdf.py)
%pip install sentence-transformers faiss-cpu langchain-groq python-dotenv pypdf


In [ ]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the
# kernel's working directory — this works whether the kernel launches
# from the repo root (like the lab script) or from the notebook's own
# folder (Jupyter's default) — then cd into it so every repo-relative
# path behaves exactly like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))


from dotenv import load_dotenv  # noqa: E402
load_dotenv(REPO_ROOT / ".env")  # GROQ_API_KEY lives in the repo-root .env
from embeddings.bge import BGEEmbedding  # noqa: E402
from evaluation.golden import GOLDEN_QA  # noqa: E402
from evaluation.judge import LLMJudge  # noqa: E402
from evaluation.metrics import FaithfulnessMetric  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from llms.groq import GroqLLM  # noqa: E402
from loaders.pdf import PDFLoader  # noqa: E402
from retrieval.similarity import SimilarityRetriever  # noqa: E402
from vectordb.faiss import FAISSVectorStore  # noqa: E402



## 1. Configuration — the golden set and the two variants

`GOLDEN_QA` (from `src/evaluation/golden.py`) fixes the contract: 8 questions with substantive gold answers over the SD-08 invoice PDFs. Variant **A** retrieves `top_k = 3` context chunks; variant **B** retrieves `top_k = 1`. Everything else — index, questions, references, generator, judge — is shared, so any score delta is attributable to that one knob. `FAITHFULNESS_TOLERANCE = 0.15` gives the judge-based metric room to breathe.


In [ ]:
# 1. Configuration — the golden set and the two pipeline variants
INVOICE_DIR = Path("Data/SD-08-invoices")
GOLDEN_DOCS = ["sample-invoice", "Invoice_1"]  # subset of GOLDEN_QA keys
VARIANT_A_TOP_K = 3  # reference pipeline
VARIANT_B_TOP_K = 1  # the regression: single-chunk context
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"

# Faithfulness is judge-based and measures claim-support DENSITY, not answer
# completeness: a pipeline that answers less can score HIGHER (fewer claims,
# fewer chances to hallucinate). It is also noisy on a 10-question sample.
# So the regression gate tolerates a small faithfulness dip while gating the
# reference-anchored metrics (containment, cosine) strictly.
FAITHFULNESS_TOLERANCE = 0.15


## 2. Reference-based correctness (same helpers as labs 02–03)

The deterministic scoring block reused verbatim: cosine, containment, and the claim-level faithfulness judge metric. One pipeline, two scoring philosophies.


In [ ]:
# 2. Reference-based correctness (same helpers as labs 02-03)
def normalize(text: str) -> str:
    """Lowercase, strip punctuation/articles and whitespace."""
    cleaned = "".join(c.lower() for c in text if c.isalnum() or c.isspace())
    words = [w for w in cleaned.split() if w not in ("a", "an", "the")]
    return " ".join(words)


def reference_contained(answer: str, reference: str) -> bool:
    """True when the normalized gold answer appears inside the answer."""
    return normalize(reference) in normalize(answer)


def cosine_similarity(a: list[float], b: list[float]) -> float:
    """Cosine similarity between two vectors (0.0 if either is zero)."""
    dot = sum(x * y for x, y in zip(a, b))
    na = sum(x * x for x in a) ** 0.5
    nb = sum(x * x for x in b) ** 0.5
    if na == 0.0 or nb == 0.0:
        return 0.0
    return dot / (na * nb)


## 3. One pipeline, parameterized by top_k

`run_variant(top_k)` builds the retriever + judge + generator, scores the golden set, and returns the per-question rows plus means. A and B are the *same function* with different arguments — that is what makes this a controlled A/B rather than two ad-hoc scripts.


In [ ]:
# 3. One pipeline, parameterized by top_k — A and B differ in ONE knob
def build_indexes() -> dict[str, SimilarityRetriever]:
    """Per-doc index: {doc_key: retriever over that invoice's pages}."""
    embedder = BGEEmbedding(model_name=BGE_MODEL_NAME)
    retrievers: dict[str, SimilarityRetriever] = {}
    for doc_key in GOLDEN_DOCS:
        pdf_path = INVOICE_DIR / f"{doc_key}.pdf"
        pages = PDFLoader(str(pdf_path)).load()
        texts = [p.page_content for p in pages]
        vectors = embedder.embed_documents(texts)
        chunks = [
            Document(page_content=t, metadata={"doc": doc_key, "page": i})
            for i, t in enumerate(texts)
        ]
        store = FAISSVectorStore(embedding=embedder)
        store.add(chunks, embeddings=vectors)
        retrievers[doc_key] = SimilarityRetriever(store, top_k=20)
    return retrievers


def run_variant(retrievers: dict[str, SimilarityRetriever], top_k: int,
                llm, judge, faithfulness) -> dict[str, float]:
    """Run the golden set through the pipeline with a given ``top_k``.

    Returns mean scores per metric over the golden questions plus the
    per-question rows (doc, question, contained) for eyeballing.
    """
    per_metric: dict[str, list[float]] = {"faithfulness": [], "contained": [],
                                          "cosine": []}
    rows: list[dict] = []
    for doc_key in GOLDEN_DOCS:
        retriever = retrievers[doc_key]
        for question, reference in GOLDEN_QA[doc_key]:
            docs = retriever.retrieve(question)[:top_k]
            context = "\n\n".join(d.page_content for d in docs)
            answer = llm.invoke(
                f"Context:\n{context}\n\nQuestion: {question}\n\n"
                "Answer concisely, quoting the exact figure or value from "
                "the context:"
            ).strip()
            per_metric["faithfulness"].append(
                faithfulness.score(question, context, answer))
            contained = reference_contained(answer, reference)
            per_metric["contained"].append(contained)
            per_metric["cosine"].append(
                cosine_similarity(judge.embed([answer])[0],
                                  judge.embed([reference])[0]))
            rows.append({"doc": doc_key, "question": question,
                         "contained": contained})
    means = {k: sum(v) / len(v) for k, v in per_metric.items()}
    means["rows"] = rows
    return means


## 4. Experiment — build once, run both variants

The index is built **once** and shared by both variants — only the retriever's `top_k` differs. Then both variants run over the golden set and each is scored on faithfulness, contained, and cosine.


In [ ]:
# 4. Experiment — build once, run both variants
def run_experiment() -> dict:
    t0 = time.perf_counter()
    retrievers = build_indexes()
    index_s = time.perf_counter() - t0

    llm = GroqLLM(temperature=0.0)
    judge = LLMJudge()
    faithfulness = FaithfulnessMetric(judge)

    t0 = time.perf_counter()
    variant_a = run_variant(retrievers, VARIANT_A_TOP_K, llm, judge, faithfulness)
    variant_b = run_variant(retrievers, VARIANT_B_TOP_K, llm, judge, faithfulness)
    eval_s = time.perf_counter() - t0

    return {
        "docs": GOLDEN_DOCS,
        "n_questions": sum(len(GOLDEN_QA[d]) for d in GOLDEN_DOCS),
        "index_s": index_s,
        "eval_s": eval_s,
        "variant_a": variant_a,
        "variant_b": variant_b,
    }


## 5. Demo

The demo prints the A/B score table, the per-question containment rows, and the regression verdicts. Expect a subtlety: on faithfulness, B sometimes *beats* A — a single chunk is claim-support *density*, and a model that asserts less has less to contradict. The reference-anchored metrics (contained, cosine) gate strictly; faithfulness gets the tolerance band.


In [ ]:
# 5. Demo
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 07-04 — Golden-set regression (evaluation/golden.py)")
    print(f"docs {exp['docs']}, {exp['n_questions']} golden questions")
    print("=" * 66)

    a, b = exp["variant_a"], exp["variant_b"]
    print(f"\n[1] Golden-set scores (mean over {exp['n_questions']} questions):")
    print(f"    {'metric':<14} {'A (top_k=3)':>12} {'B (top_k=1)':>12}  delta")
    for key in ("faithfulness", "contained", "cosine"):
        print(f"    {key:<14} {a[key]:>12.3f} {b[key]:>12.3f}  "
              f"{a[key] - b[key]:+.3f}")

    print(f"\n[2] Per-question reference containment (eyeball the rows):")
    print(f"    {'doc':<14} {'A':>4} {'B':>4}  question")
    for ra, rb in zip(a["rows"], b["rows"]):
        print(f"    {ra['doc']:<14} {ra['contained']:>4.0f} "
              f"{rb['contained']:>4.0f}  {ra['question'][:52]}")

    print(f"\n[3] Regression gate (reference-anchored strict, faithfulness "
          f"tolerated):")
    a_f, b_f = a["faithfulness"], b["faithfulness"]
    for key in ("contained", "cosine"):
        status = "PASS" if a[key] >= b[key] else "FAIL"
        print(f"    [{status}] A >= B for {key}")
    status = "PASS" if a_f >= b_f - FAITHFULNESS_TOLERANCE else "FAIL"
    print(f"    [{status}] A >= B - {FAITHFULNESS_TOLERANCE:.2f} for faithfulness")

    print(f"\n[4] Timing: index {exp['index_s']:.1f}s, evaluate both "
          f"{exp['eval_s']:.1f}s")

    print(f"\n[5] Takeaway")
    print("    A golden set is the regression contract: the same questions,")
    print("    the same references, forever. Variant B is a realistic silent")
    print("    regression — one context chunk instead of three — and the gate")
    print("    is 'reference pipeline >= new pipeline'. Two subtleties the")
    print("    numbers teach: (1) faithfulness is claim-support DENSITY, not")
    print("    completeness — a single chunk can score higher because the")
    print("    model asserts less; judge metrics get a tolerance. (2) the")
    print("    reference-anchored metrics gate strictly — when a change")
    print("    passes unit tests but drops a golden containment/cosine score,")
    print("    this gate stops it from shipping. Always eyeball the rows.")


## 6. Verification gate

The gate enforces the regression contract: `A >= B` for contained and cosine, `A >= B - 0.15` for faithfulness, a floor that the golden set is actually answerable, and all scores in [0, 1]. A change that passes unit tests but drops a golden score is stopped here.


In [ ]:
# 6. Verification gate
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    a, b = exp["variant_a"], exp["variant_b"]

    checks.append((f"{exp['n_questions']} golden questions evaluated (>= 6)",
                   exp["n_questions"] >= 6))

    for key in ("faithfulness", "contained", "cosine"):
        checks.append((f"A {key} in [0, 1]", 0.0 <= a[key] <= 1.0))
        checks.append((f"B {key} in [0, 1]", 0.0 <= b[key] <= 1.0))

    # The regression contract: A beats B on every metric. Reference-anchored
    # metrics gate strictly; the judge-based faithfulness metric gets the
    # tolerance band (claim-support density is not monotonic in context and
    # the judge is noisy on a 10-question sample).
    for key in ("contained", "cosine"):
        checks.append((f"regression gate: A >= B for {key}", a[key] >= b[key]))
    checks.append((
        f"regression gate: A >= B - {FAITHFULNESS_TOLERANCE} for faithfulness",
        a["faithfulness"] >= b["faithfulness"] - FAITHFULNESS_TOLERANCE))

    # A floor so the golden set is doing real work (not all-zeros).
    checks.append(("reference pipeline finds some answers (A contained > 0)",
                   a["contained"] > 0.0))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

Indexing two small PDFs is fast; the 16 judge-scored generations (8 per variant) take a couple of minutes. `exp` holds both variants plus timing.


In [ ]:
exp = run_experiment()


### Demo — the artifact


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces.


In [ ]:
verify_gate(exp)
